# E1. Social Media 30-Day Content Calendar Notebook

> **Related Module**: [E7 Cross-Channel Coordination](../paths/e-social-media/e7-social-media-cross-channel.md)
>
> **Features**: Input brand info → AI generates a 30-day cross-platform content plan
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/e1-social-content-calendar.ipynb)

In [ ]:
!pip install -q openai pandas

## 1. Input Brand Information

In [ ]:
import os, json
from openai import OpenAI
import pandas as pd
from datetime import datetime, timedelta

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')
client = OpenAI(api_key=OPENAI_API_KEY)

# === Replace with your brand ===
BRAND = {
    'name': 'SoundPeak',
    'category': 'Wireless Earbuds',
    'target_audience': '25-40 year old professionals and fitness enthusiasts',
    'brand_voice': 'Professional yet approachable, tech-savvy',
    'platforms': ['Instagram', 'TikTok', 'Pinterest'],
    'posting_frequency': {'Instagram': 4, 'TikTok': 5, 'Pinterest': 3},  # posts per week
    'key_products': ['SoundPeak Pro ANC Earbuds', 'SoundPeak Sport Earbuds'],
    'upcoming_events': ['Summer Sale (July)', 'Back to School (August)', 'Prime Day (July 15)']
}

START_DATE = datetime.now().date()
print(f'Brand: {BRAND["name"]}')
print(f'Platforms: {", ".join(BRAND["platforms"])}')
print(f'Calendar start: {START_DATE}')
print(f'Calendar end: {START_DATE + timedelta(days=30)}')

## 2. AI-Generated 30-Day Content Calendar

In [ ]:
prompt = f"""You are a social media content strategist for e-commerce brands.

Brand: {BRAND['name']}
Category: {BRAND['category']}
Audience: {BRAND['target_audience']}
Voice: {BRAND['brand_voice']}
Platforms: {', '.join(BRAND['platforms'])}
Weekly frequency: {json.dumps(BRAND['posting_frequency'])}
Products: {', '.join(BRAND['key_products'])}
Upcoming events: {', '.join(BRAND['upcoming_events'])}
Start date: {START_DATE}

Generate a 30-day content calendar. Return JSON array where each item is:
{{
  "day": 1-30,
  "date": "YYYY-MM-DD",
  "platform": "Instagram/TikTok/Pinterest",
  "content_type": "Reel/Carousel/Story/TikTok/Pin",
  "topic": "brief topic",
  "caption_hook": "first line of caption or video hook",
  "hashtags": "5 relevant hashtags",
  "content_pillar": "Education/Entertainment/Promotion/UGC/Behind-the-scenes"
}}

Rules:
- Mix content pillars (40% education, 25% entertainment, 20% promotion, 15% UGC/BTS)
- Never post promotions 2 days in a row
- TikTok hooks must create information gaps
- Instagram carousels for educational content
- Pinterest pins should be SEO-optimized
- Include upcoming events naturally
- Generate exactly 30 days of content"""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}],
    response_format={'type': 'json_object'},
    max_tokens=4000
)

result = json.loads(response.choices[0].message.content)
calendar = result if isinstance(result, list) else result.get('calendar', result.get('content_calendar', []))
print(f'Generated {len(calendar)} content items')

## 3. Calendar Visualization

In [ ]:
df = pd.DataFrame(calendar)
print(f'=== 30-Day Content Calendar: {BRAND["name"]} ===')
print(f'\nPlatform distribution:')
print(df['platform'].value_counts().to_string())
print(f'\nContent pillar distribution:')
if 'content_pillar' in df.columns:
    print(df['content_pillar'].value_counts().to_string())

print(f'\n{"="*70}')
for _, row in df.iterrows():
    platform_emoji = {'Instagram': '📸', 'TikTok': '🎵', 'Pinterest': '📌'}.get(row.get('platform', ''), '📄')
    pillar_emoji = {'Education': '📚', 'Entertainment': '🎭', 'Promotion': '🏷️', 'UGC': '👥', 'Behind-the-scenes': '🎬'}.get(row.get('content_pillar', ''), '')
    print(f"Day {row.get('day', '?'):2d} | {platform_emoji} {row.get('platform', ''):10s} | {row.get('content_type', ''):10s} | {pillar_emoji} {row.get('topic', '')[:40]}")
    print(f"       Hook: {row.get('caption_hook', '')[:60]}")
    print()

## 4. Export

In [ ]:
df.to_csv('content_calendar_30d.csv', index=False)
print(f'Exported {len(df)} items to content_calendar_30d.csv')

# Export by platform
for platform in BRAND['platforms']:
    platform_df = df[df['platform'] == platform]
    filename = f'calendar_{platform.lower()}.csv'
    platform_df.to_csv(filename, index=False)
    print(f'  {platform}: {len(platform_df)} posts → {filename}')

print('\nNext: Use these as briefs for content creation with AI image/video tools')